# 03 缺值處理與 PCA（相關係數矩陣）

**用途**：檢查缺值並排除連江縣；對各年度標準化後做 PCA，輸出解釋變異、陡坡圖、累積變異與雙標圖。

| 項目 | 說明 |
|---|---|
| 輸入 | pivot_table_age_std |
| 輸出 | pivot_table_cleaned、scaled_data、pca_corr、loadings/scores、陡坡圖與雙標圖 |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 116–133（Update 05/29 最終版） |


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

## Data Cleaning

* 排除連江縣

In [ ]:
# 檢查 NaN values 狀況
# Check for rows (counties) with NaN values
rows_with_nan = pivot_table_age_std[pivot_table_age_std.isna().any(axis=1)].index.tolist()

# Check for columns (years) with NaN values
cols_with_nan = pivot_table_age_std.columns[pivot_table_age_std.isna().any(axis=0)].tolist()

print(f"Rows (Counties) with NaN values: {rows_with_nan}")
print(f"Columns (Years) with NaN values: {cols_with_nan}")

# Display the exact locations (Row, Column) where values are missing
if rows_with_nan or cols_with_nan:
    print("\nDetailed NaN locations:")
    nan_locations = pivot_table_age_std.isna().stack()
    nan_locations = nan_locations[nan_locations].index.tolist()
    for row, col in nan_locations:
        print(f"- County: {row}, Year: {col}")
else:
    print("No NaN values found in the table.")

連江縣在 1979 ~ 2000 年間有 11 次出現 NaN

In [ ]:
pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

## PCA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

### PCA using corr

In [ ]:
# ==========================================
# 1. 資料標準化
# ==========================================
scaler = StandardScaler()
scaled_data = scaler.fit_transform(pivot_table_cleaned)

In [ ]:
# ==========================================
# 2. 執行 PCA
# ==========================================
pca_corr = PCA()
pca_corr.fit(scaled_data)

# ✨ 使用 f"{x:.2%}" 自動將小數轉換為百分比並四捨五入至小數點後兩位
explained_variance_pct = [f"{x:.2%}" for x in pca_corr.explained_variance_ratio_[:7]]
cumulative_variance_pct = [f"{x:.2%}" for x in np.cumsum(pca_corr.explained_variance_ratio_)[:7]]

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前7個):", explained_variance_pct)
print("累積解釋變異比 (前7個):", cumulative_variance_pct)
print("-" * 38)

In [ ]:
# 原 Colab 於後面的雙標圖 cell 才定義 loadings，這裡先定義以便依序執行
loadings = pca_corr.components_.T[:, :2]   # 前兩個主成分的負荷量

In [ ]:
loadings

In [ ]:
df_loadings_corr = pd.DataFrame(
    loadings,
    index=pivot_table_cleaned.columns,
    columns=['PC1', 'PC2']
)
# Round to two decimal places
df_loadings_corr = df_loadings_corr.round(2)
display(df_loadings_corr)

#### Scree Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 取得特徵值 (λ)
eigenvalues = pca_corr.explained_variance_

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(eigenvalues) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製陡坡圖的折線與點
plt.plot(
    pc_num,
    eigenvalues,
    marker='o',
    linestyle='-',
    color='#2b5c8f',
    linewidth=2,
    markersize=6,
    label='Eigenvalue'
)

# 加上傳統轉折點（Elbow）的輔助標記或 Kaiser 準則線
# 註：如果使用的是「相關係數矩陣(Scaled)」，通常會畫一條 λ = 1 的水平線作為篩選基準
plt.axhline(y=1, color='r', linestyle='--', alpha=0.7, label='Kaiser Criterion (λ = 1)')

plt.title('PCA Scree Plot - Correlation Matrix', fontsize=16)
plt.xlabel('Principal Component', fontsize=13)
plt.ylabel('Eigenvalue (λ)', fontsize=13)

# 讓 X 軸的刻度整齊顯示每一個 PC 編號
plt.xticks(pc_num)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Variance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算累積解釋變異量比率 (乘以 100 轉為百分比)
cum_variance_ratio = np.cumsum(pca_corr.explained_variance_ratio_) * 100

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_variance_ratio) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製累積變異量的折線與點
plt.plot(
    pc_num,
    cum_variance_ratio,
    marker='s',          # 使用方形標記區隔之前的陡坡圖
    linestyle='-',
    color='#e06666',     # 使用顯眼的暖色系
    linewidth=2,
    markersize=6,
    label='Cumulative Explained Variance'
)

# ✨ 加上 80% 的基準輔助線 (通常實務上會看 80% 或 90% 的斷點)
plt.axhline(y=90, color='gray', linestyle='--', alpha=0.7, label='90% Threshold')

# 標註前幾個 PC 的具體數值（讓圖表更直覺）
# 這邊標註前 3 個成分作為範例
for i in range(3):
    plt.annotate(
        f"{cum_variance_ratio[i]:.1f}%",
        (pc_num[i], cum_variance_ratio[i]),
        textcoords="offset points",
        xytext=(0,10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# 圖表細節設定
plt.title('PCA (Correlation) Cumulative Explained Variance', fontsize=16)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

# 讓 X、Y 軸的刻度更貼切
plt.xticks(pc_num)
plt.ylim(0, 105) # Y 軸範圍設在 0~105% 比較有留白美感

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_cumulative_variance_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Biplot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 準備必要數據
# --------------------------------------------------
# (A) 取得 PC Score (縣市投影點在 PC1 和 PC2 的坐標)
scores = pca_corr.transform(scaled_data)
pc1_scores = scores[:, 0]
pc2_scores = -scores[:, 1]
county_names = pivot_table_cleaned.index # 縣市名稱

# (B) 取得 PC Loadings (原始欄位在 PC1 和 PC2 的載荷/箭頭向量)
loadings = pca_corr.components_.T # 轉置成 (num_features, num_components)
pc1_loadings = loadings[:, 0]
pc2_loadings = -loadings[:, 1]
feature_names = pivot_table_cleaned.columns # 原始欄位名稱 (例如：年份)

# (C) 解釋變異量百分比 (對齊前面 R 的 summary 結果)
var1_pct = pca_corr.explained_variance_ratio_[0] * 100
var2_pct = pca_corr.explained_variance_ratio_[1] * 100

# ✨ (D) 計算 Loadings 箭頭的縮放因子 (Scale Factor)
# Biplot 核心技巧：讓 Loading 箭頭長度能夠根據 Score 點的級距動態調整。
# 這裡將 Loadings 最大絕對值拉伸到和 Scores 最大絕對值同一個數量級，再乘上 0.9 (填滿 90% 畫面)
max_score = np.max([np.abs(pc1_scores).max(), np.abs(pc2_scores).max()])
max_loading = np.max([np.abs(pc1_loadings).max(), np.abs(pc2_loadings).max()])
scale_factor = (max_score / max_loading) * 0.9

# 2. 開始繪圖
# --------------------------------------------------
# 確保全域中文字型已設定 (若沒有 prop 變數，請重新執行 Notebook 最上方的字型註冊區塊)
if 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

fig, ax = plt.subplots(figsize=(14*0.8, 11*0.8))

# 📈 (A) 繪製 Score Plot (縣市投影點)
ax.scatter(
    pc1_scores, pc2_scores,
    c='#2b5c8f', alpha=0.6, s=70, edgecolors='white',
    label='縣市投影點'
)

# ✍️ 標註縣市名稱 (中文字)
# 設定小的 xytext 偏移量，避免文字貼在點上
for i, txt in enumerate(county_names):
    ax.annotate(
        txt, (pc1_scores[i], pc2_scores[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#1f3c5f', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的年份箭頭向量)
arrow_color = '#e06666' # 暖色調，與藍點形成對比
for i in range(len(feature_names)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings[i] * scale_factor
    end_y = pc2_loadings[i] * scale_factor

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color, alpha=0.5, linewidth=1,
        head_width=max_score*0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (年份) - 根據箭頭方向微調文字對齊方式，避免疊加
    # 文字放在箭頭終點附近 (加 2% 偏移量)
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(feature_names[i]),
        color=arrow_color, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
# 畫出 PC1、PC2 的原點輔助線
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

# 設定標題和座標軸標籤 (包含解釋變異量比例)
ax.set_title(
    f'PCA Biplot：肺癌年齡標準化發生率 (Correlation)',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1 ', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

# 確保座標軸比例 1:1，避免形變 (這在 PCA 報告中非常重要)
# 若Scores和Loadings級距差異太大，設定 'equal' 有時會導致圖表縮得太小，可依喜好註解掉
# ax.set_aspect('equal')

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_corr_biplot_counties_by_years.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

# 1. 準備必要數據：將 pivot_table_cleaned 轉置，讓年份成為觀測值(rows)，縣市成為變數(columns)
# --------------------------------------------------
data_for_pca_swapped = pivot_table_cleaned.T

# 2. 標準化轉置後的數據 (與 pca_corr 的邏輯一致，使用 StandardScaler)
# --------------------------------------------------
scaler_swapped = StandardScaler()
scaled_data_swapped = scaler_swapped.fit_transform(data_for_pca_swapped)

# 3. 執行 PCA
# --------------------------------------------------
pca_swapped = PCA(n_components=2)
pca_swapped.fit(scaled_data_swapped)

# (A) 取得 PC Score (年份投影點在 PC1 和 PC2 的坐標)
scores_swapped = pca_swapped.transform(scaled_data_swapped)
pc1_scores_swapped = scores_swapped[:, 0]
pc2_scores_swapped = -scores_swapped[:, 1] # 保持與之前 PC2 軸一致的負號翻轉
year_names = data_for_pca_swapped.index # 年份作為點的標籤

# (B) 取得 PC Loadings (縣市在 PC1 和 PC2 的載荷/箭頭向量)
loadings_swapped = pca_swapped.components_.T # 轉置成 (num_features, num_components)
pc1_loadings_swapped = loadings_swapped[:, 0]
pc2_loadings_swapped = -loadings_swapped[:, 1] # 保持與之前 PC2 軸一致的負號翻轉
county_names_swapped = data_for_pca_swapped.columns # 縣市作為箭頭的標籤

# (C) 解釋變異量百分比
var1_pct_swapped = pca_swapped.explained_variance_ratio_[0] * 100
var2_pct_swapped = pca_swapped.explained_variance_ratio_[1] * 100

# (D) 計算 Loadings 箭頭的縮放因子
max_score_swapped = np.max([np.abs(pc1_scores_swapped).max(), np.abs(pc2_scores_swapped).max()])
max_loading_swapped = np.max([np.abs(pc1_loadings_swapped).max(), np.abs(pc2_loadings_swapped).max()])
scale_factor_swapped = (max_score_swapped / max_loading_swapped) * 0.9

# 4. 開始繪圖
# --------------------------------------------------
# 確保中文字型已設定
if 'font_name' in globals():
    plt.rcParams['font.family'] = font_name

fig, ax = plt.subplots(figsize=(14 * 0.8, 11 * 0.8))

# ➗️ (A) 繪製 Score Plot (年份投影點)
ax.scatter(
    pc1_scores_swapped, pc2_scores_swapped,
    c='#e06666', alpha=0.6, s=70, edgecolors='white',
    label='年份投影點'
)

# ✍️ 標註年份 (數字)
for i, txt in enumerate(year_names):
    ax.annotate(
        txt, (pc1_scores_swapped[i], pc2_scores_swapped[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#b30000', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的縣市箭頭向量)
arrow_color_swapped = '#2b5c8f' # 與點形成對比的顏色
for i in range(len(county_names_swapped)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings_swapped[i] * scale_factor_swapped
    end_y = pc2_loadings_swapped[i] * scale_factor_swapped

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color_swapped, alpha=0.5, linewidth=1,
        head_width=max_score_swapped * 0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (縣市) - 根據箭頭方向微調文字對齊方式
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(county_names_swapped[i]),
        color=arrow_color_swapped, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

ax.set_title(
    f'主成分分析雙標圖',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1 ', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

plt.savefig('pca_corr_biplot_years_by_counties.png', dpi=300, bbox_inches='tight')
plt.show()